# second-ear: continue the where-only training on Colab
Run the cells top to bottom (Runtime > Run all). Pick a T4 GPU first: Runtime > Change runtime type.

## 1. Check the GPU and connect Google Drive
Shows which graphics card Colab gave you, then asks permission to open your Drive.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

## 2. Settings
Where your files live on Drive, which code branch to use, which models to train, and which feature set (`DATA`) and Drive subfolder (`FEATURES`) to use.

In [ ]:
DRIVE = '/content/drive/MyDrive/second-ear'
BRANCH = 'feature/colab-train'
MODELS = ['binaural', 'left', 'right']
DATA = 'binaural_v2'    # feature set name, e.g. 'binaural_v3'
FEATURES = 'features'   # Drive subfolder holding the shards (and scenes tars)

## 3. Get the code and install it
Downloads the project from GitHub (or updates it) and installs it. Colab already has PyTorch, so we do not reinstall it. Colab's Python is newer than the project asks for, so we skip the version check and install the other libraries by hand.

In [ ]:
import os
os.environ['DRIVE'] = DRIVE
os.environ['BRANCH'] = BRANCH
if os.path.isdir('/content/second-ear/.git'):
    !git -C /content/second-ear fetch origin $BRANCH && git -C /content/second-ear checkout $BRANCH && git -C /content/second-ear pull origin $BRANCH
else:
    !git clone --branch $BRANCH https://github.com/JohnFok119/second-ear.git /content/second-ear
%cd /content/second-ear
!pip install -q --no-deps --ignore-requires-python -e .
!pip install -q numpy scipy soundfile librosa pyroomacoustics spaudiopy sofar matplotlib pyyaml tqdm requests "tensorboard>=2.21.0"
!python -c "import torch, secondear.train.heatmap; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 4. Unpack the features onto the Colab disk
Copies each float16 shard from Drive and unpacks it. If there are `scenes_*.tar` files next to them (the raw val/test audio and labels, needed for the Score step), those are unpacked too. Anything already unpacked is skipped, so re-running is quick.

In [ ]:
import glob, os, tarfile
DEST = f'/content/second-ear/data/features/{DATA}'
SCENES = f'/content/second-ear/data/{DATA}'

def unpack(pattern, dest):
    os.makedirs(dest, exist_ok=True)
    tars = sorted(glob.glob(pattern))
    for s in tars:
        marker = os.path.join(dest, '.extracted_' + os.path.basename(s))
        if os.path.exists(marker):
            print('skip', os.path.basename(s))
            continue
        print('extract', os.path.basename(s))
        with tarfile.open(s) as t:
            t.extractall(dest)
        open(marker, 'w').close()
    return tars

shards = unpack(f'{DRIVE}/{FEATURES}/feat_f16_*.tar', DEST)
assert shards, f'no shards found in {DRIVE}/{FEATURES}'
print(len(glob.glob(DEST + '/train/*.npz')), 'train clips,', len(glob.glob(DEST + '/val/*.npz')), 'val clips')
if not unpack(f'{DRIVE}/{FEATURES}/scenes_*.tar', SCENES):
    print(f'note: no scenes_*.tar in {DRIVE}/{FEATURES}; the Score step will be skipped')

## 5. Save progress to Drive
Makes the checkpoints and runs folders live on Drive, so a disconnect does not lose them. Your uploaded `where_binaural` files are picked up from here.

In [ ]:
import os, shutil
for name in ('checkpoints', 'runs'):
    target = f'{DRIVE}/{name}'
    link = f'/content/second-ear/{name}'
    os.makedirs(target, exist_ok=True)
    if os.path.islink(link):
        continue
    if os.path.isdir(link):
        shutil.rmtree(link)
    os.symlink(target, link)
print(os.listdir(f'{DRIVE}/checkpoints'))

## 6. Train
Trains each model in turn. A model with a `done.txt` is skipped; one with a `last.pt` carries on from where it stopped. After a disconnect, just run all cells again.

In [ ]:
import os, subprocess
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
for m in MODELS:
    ck = f'{DRIVE}/checkpoints/where_{m}'
    if os.path.exists(f'{ck}/done.txt'):
        print(m, ': already done, skipping')
        continue
    cmd = ['python', 'scripts/train_heatmap.py', '--config', f'configs/where_{m}.yaml', '--device', 'cuda', '--ram-f16']
    if os.path.exists(f'{ck}/last.pt'):
        cmd.append('--resume')
    print(m, ':', ' '.join(cmd), flush=True)
    log = f'{DRIVE}/runs/where_{m}_train.log'
    with open(log, 'a', encoding='utf-8') as fh:
        p = subprocess.Popen(cmd, cwd='/content/second-ear', stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='')
            fh.write(line)
            fh.flush()
        rc = p.wait()
    if rc != 0:
        raise RuntimeError(f'{m} training failed with exit code {rc}; see {log}')
    os.makedirs(ck, exist_ok=True)
    open(f'{ck}/done.txt', 'w').write('done\n')
print('all requested models finished')

## 7. Score
For each finished model that has not been scored yet: picks the best decoding threshold on the val scenes, then evaluates on the test scenes (the two-ear model also with the left and right ear muted). The small result folders are copied to `results/` on Drive. Needs the `scenes_*.tar` from step 4.

In [ ]:
import glob, json, os, shutil, subprocess
from secondear.train.heatmap import load_config
ROOT = '/content/second-ear'

def run(cmd, log):
    print(' '.join(cmd), flush=True)
    with open(log, 'a', encoding='utf-8') as fh:
        p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='')
            fh.write(line)
            fh.flush()
        rc = p.wait()
    if rc != 0:
        raise RuntimeError(f'failed with exit code {rc}; see {log}')

os.makedirs(f'{DRIVE}/runs', exist_ok=True)
os.makedirs(f'{DRIVE}/results', exist_ok=True)
for m in MODELS:
    if not os.path.exists(f'{DRIVE}/checkpoints/where_{m}/done.txt'):
        print(m, ': not trained yet, skipping')
        continue
    if os.path.exists(f'{DRIVE}/results/where_{m}/test_done.txt'):
        print(m, ': already scored, skipping')
        continue
    if not os.path.isdir(f'{SCENES}/val'):
        print(m, f': {SCENES}/val is missing (upload scenes_*.tar), skipping')
        continue
    cfg_path = f'configs/where_{m}.yaml'
    cfg = load_config(f'{ROOT}/{cfg_path}')
    exp = cfg['experiment']
    base = ['python', 'scripts/pick_threshold.py', '--config', cfg_path, '--split', 'val', '--device', 'cuda']
    run(base, f'{DRIVE}/runs/where_{m}_pick_threshold.log')
    scored = os.path.isdir(f'{SCENES}/test')
    if scored:
        with open(f'{ROOT}/results/{exp}/threshold_val.json') as fh:
            chosen = json.load(fh)['chosen']
        ev = ['python', 'scripts/eval_heatmap.py', '--config', cfg_path, '--split', 'test',
              '--device', 'cuda', '--threshold', str(chosen)]
        run(ev, f'{DRIVE}/runs/where_{m}_eval.log')
        if cfg['ears'] == 'both':
            for side in ('left', 'right'):
                run(ev + ['--mute', side], f'{DRIVE}/runs/where_{m}_mute_{side}_eval.log')
    else:
        print(m, f': {SCENES}/test is missing, skipping the test evaluations')
    folders = [f'results/{exp}'] + [os.path.relpath(d, ROOT) for pat in (f'{exp}_thr*', f'{exp}_mute_*')
                                    for d in glob.glob(f'{ROOT}/results/{pat}')]
    for f in folders:
        shutil.copytree(f'{ROOT}/{f}', f'{DRIVE}/{f}', dirs_exist_ok=True)
    if scored:
        open(f'{DRIVE}/results/{exp}/test_done.txt', 'w').write('done
')
    print(m, ': copied', folders)
print('scoring finished')